# Amazon CloudWatch와 AWS CloudTrail을 사용한 AgentCore Gateway 관찰성 구성

* Amazon CloudWatch는 AgentCore Gateway의 실시간 성능 모니터링과 운영 문제 해결에 중점을 두며, 지연 시간, 오류율, 사용 패턴에 대한 상세 지표와 로그를 제공합니다. 
* AWS CloudTrail은 Gateway와 관련된 API 호출 및 사용자 작업의 전체 기록을 저장하여 보안, 규정 준수, 감사에 중점을 둡니다. 

두 서비스를 함께 사용하면 프로덕션 환경에서 AgentCore Gateway를 관리하기 위한 포괄적인 관찰성 및 거버넌스 프레임워크를 구축할 수 있습니다.

**Amazon CloudWatch**

주로 AgentCore Gateway 데이터 플레인 상호 작용을 기록합니다. Gateway 도구 목록 조회(tools/list), Gateway 도구 호출(tools/call), Gateway 도구 검색(tools/call)

| 구성 요소 유형 | 설명 | 
| --- | --- | 
| 지표 | 성능 및 운영 데이터 | 
| 추적, 스팬, 요청 | 요청 경로 추적 | 
| 애플리케이션 로그 | 데이터 플레인 운영 로그 | 

**Amazon CloudTrail**

AgentCore Gateway 컨트롤 플레인(CreateGateway, ListGateway, DeleteGateaway 등)과 데이터 플레인 상호 작용(InvokeGateway 등)을 기록합니다.

| 구성 요소 유형 | 설명 | 
| --- | --- | 
| 관리 이벤트 | 컨트롤 플레인 요청의 자격 증명 정보를 포함하며, 추적 생성 시 기본적으로 활성화됩니다. | 
| 데이터 이벤트 | 리소스에서 또는 리소스를 대상으로 수행된 작업에 대한 정보입니다. 데이터 이벤트는 대량으로 발생하는 경우가 많으므로 명시적으로 활성화해야 하며 추가 요금이 부과됩니다. | 



## AgentCore Gateway 생성 및 AWS CloudTrail 모니터링 

#### 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Jupyter notebook (Python kernel)
* uv
* AWS 자격 증명
* Amazon Cognito

In [ ]:
!pip install --force-reinstall -U -r requirements.txt

In [ ]:
# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
import os

# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-west-2")  # AWS 리전 설정

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우 대체 경로 사용(예: Jupyter)

# utils.py가 있는 디렉터리로 이동(한 단계 위)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# 이제 utils를 가져올 수 있음
import utils

#### Lambda 함수 생성

**아래 출력에서 Lambda 함수 ARN을 기록해 둡니다.**

In [ ]:
#### MCP 도구로 변환할 샘플 AWS Lambda 함수를 생성하고 Lambda ARN 기록

lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp["exit_code"] == 0:
        print(
            "NOTE DOWN: Lambda function created with ARN: ",
            lambda_resp["lambda_function_arn"],
        )
    else:
        print(
            "NOTE DOWN: Lambda function creation failed with message: ",
            lambda_resp["lambda_function_arn"],
        )

In [ ]:
#### Gateway가 수임할 IAM 역할 생성
import utils

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

#### Gateway 인바운드 권한 부여를 위한 Amazon Cognito 풀 생성

In [ ]:
# Cognito User Pool 생성
import os
import boto3
import time
from botocore.exceptions import ClientError

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"},
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# 검색 URL 가져오기
cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"
print(cognito_discovery_url)

#### 인바운드 권한 부여용 Amazon Cognito Authorizer를 사용하여 Gateway 생성

In [ ]:
# CMK 없이 Cognito Authorizer로 CreateGateway를 수행하고 이전 단계에서 생성한 Cognito User Pool 사용
gateway_client = boto3.client("bedrock-agentcore-control", region_name=os.environ["AWS_DEFAULT_REGION"])
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            client_id
        ],  # 클라이언트는 Cognito에 구성된 ClientId와 반드시 일치해야 함. 예: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name="DemoGWforLambda",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway 생성/목록 조회/가져오기/삭제 권한이 있어야 함
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with AWS Lambda target type",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 가져오기
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

#### AWS Lambda 대상을 생성하고 MCP 도구로 변환

<font color="red"> **참고: Lambda 함수 ARN을 위에서 기록한 값으로 바꾸세요.** </font>

In [ ]:
# 아래 AWS Lambda 함수 ARN 바꾸기
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": "<Lambda_ARN_noted_above>",  # 위에서 기록한 AWS Lambda 함수 ARN으로 바꾸기
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"orderId": {"type": "string"}},
                            "required": ["orderId"],
                        },
                    },
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"orderId": {"type": "string"}},
                            "required": ["orderId"],
                        },
                    },
                ]
            },
        }
    }
}

credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
targetname = "LambdaUsingSDK"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description="Lambda Target using SDK",
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config,
)

## Amazon CloudWatch를 사용하여 AgentCore Gateway 관찰성 구성

### AgentCore Gateway 애플리케이션 로그 구성

AgentCore Gateway 애플리케이션 로그에 오류가 표시되는 시나리오는 다음과 같습니다.
- Gateway 실행 역할이 bedrock-agentcore를 신뢰하지 않는 상태에서 MCP Request tools/call을 수행하는 경우
- Gateway 실행 역할에 대상과 연결된 CredentialProviderArn에 대한 올바른 권한이 없는 상태에서 MCP Request tools/call을 수행하는 경우
- Gateway 실행 역할에 Lambda 함수 대상을 호출할 올바른 권한이 없는 상태에서 MCP Request tools/call을 수행하는 경우
- MCP Request에 권한 부여 헤더가 없는 경우
- MCP Request의 전달자 토큰이 유효하지 않은 경우(예: 만료됨, 유효하지 않음, 허용되지 않은 클라이언트 ID)
- 존재하지 않는 도구에 MCP Request tools/call을 수행하는 경우

#### 0단계: Vended Log 전송을 위한 새 로그 그룹 생성

**CloudWatch 로그 그룹 이름을 기록해 둡니다.**

In [ ]:
import boto3

# CloudWatch Logs 클라이언트 초기화
logs_client = boto3.client("logs", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)

cloudwatch_log_group = ""

# 로그 그룹 이름 정의
log_group_name = "/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/" + gatewayID
# AWS 계정 ID 가져오기
account_id = sts_client.get_caller_identity()["Account"]
log_group_arn = f"arn:aws:logs:{REGION}:{account_id}:log-group:{log_group_name}:*"
print(f"Log Group ARN (constructed): {log_group_arn}")

try:
    # 로그 그룹 생성
    logs_client.create_log_group(logGroupName=log_group_name)
    print(f"NOTE DOWN:Successfully created log group: {log_group_name}")
    cloudwatch_log_group = log_group_name

except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"Log group {log_group_name} already exists")
except Exception as e:
    print(f"Error creating log group: {e}")

#### 1단계: 로그 전송 소스 생성

In [ ]:
# PutDeliverySource를 사용하여 실제로 로그를 전송하는 리소스를 나타내는 논리 객체인 전송 소스 생성
gateway_name = create_response["name"]
print(gateway_name)
delivery_source_response = logs_client.put_delivery_source(
    name=f"{gateway_name}-logs-source",
    logType="APPLICATION_LOGS",  # Bedrock 전용
    resourceArn=create_response["gatewayArn"],
)

#### 2단계: 전송 대상 생성 

In [ ]:
# 전송 대상은 CloudWatch Logs의 로그 그룹, Amazon S3 버킷, Firehose 전송 스트림 또는 X-Ray를 나타낼 수 있으며 여기서는 CloudWatch 로그 그룹 사용

delivery_destination_response = logs_client.put_delivery_destination(
    name="bedrock-agentcore-gw-destination",
    deliveryDestinationType="CWL",
    deliveryDestinationConfiguration={"destinationResourceArn": log_group_arn},
    outputFormat="json",
)
print(delivery_destination_response["deliveryDestination"]["arn"])

#### 3단계: 로그 전송 생성(소스와 대상 연결)

In [ ]:
# 소스와 대상을 연결하여 로그 전송 생성. 전송은 이미 생성한 논리 전송 소스와 논리 전송 대상 간의 연결
delivery_response = logs_client.create_delivery(
    deliverySourceName=f"{gateway_name}-logs-source",
    deliveryDestinationArn=delivery_destination_response["deliveryDestination"]["arn"],
    recordFields=[
        "resource_arn",
        "event_timestamp",
        "body",
        "account_id",
        "timestamp",
        "trace_id",
        "span_id",
        "request_id",
        "gateway_id",
    ],
)

In [ ]:
time.sleep(10)

#### 4단계: AWS Console에서 확인

* AWS Console에서 [Amazon Bedrock AgentCore](https://console.aws.amazon.com/bedrock-agentcore/) 서비스로 이동합니다.
* AWS 리전이 올바른지 확인합니다.
* **Gateways**를 선택합니다.
* 생성한 Gateway를 선택합니다.
* **Log deliveries and tracing**에 항목이 표시되는지 확인합니다.

![로그 전송](images/24-amazon-bedrock-agentcore-gw.png)

### Vended Logs: AgentCore Gateway에서 tools/list 작업 호출

#### Gateway 인바운드 인증 토큰 가져오기

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes"
)
token_response = utils.get_token(user_pool_id, client_id, client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

#### Amazon CloudWatch에서 Live Tailing 활성화

In [ ]:
print(cloudwatch_log_group)

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다.
* **Log Groups**로 이동합니다.
* 위 출력에 표시된 로그 그룹을 선택합니다. 예: **/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/gatewayID**
* **Start tailing**을 클릭합니다.

<img src="images/2-cloudwatch-live-tail.png" width="60%">

#### AgentCore Gateway에서 tools/list 호출

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

with client:
    # Gateway에서 모든 도구 기능 가져오기
    tools = client.list_tools_sync()
    for tool in tools:
        print(tool.tool_name)

#### 백엔드에서는 어떤 일이 발생하나요?

**위 작업에 해당하는 로그 메시지 8개가 생성되며, Trace ID 3개, Request ID 3개, Span ID 3개가 포함됩니다.**

**Vended Logs에서 `trace_id`, `span_id`, `request_id` 등의 세부 정보를 보여 주는 흐름 순서입니다.** <br/>
**`gateway_id`도 확인할 수 있습니다.**

**Trace ID:** 사용자 대화 처리, 오케스트레이션, 도구 호출의 전체 흐름을 포함하는 전체 트랜잭션 또는 에이전트 워크플로를 나타냅니다.

**Request ID:** 추적 컨텍스트 내에서 AgentCore Gateway에 수행된 특정 요청을 식별합니다. 하나의 추적에 여러 Request ID가 존재할 수 있으며, 이는 트랜잭션 중 서로 다른 API 호출 또는 Gateway 이벤트를 나타냅니다.

**Span ID:** 특정 요청에 대해 수행된 특정 작업 또는 연산을 나타냅니다. 각 요청은 여러 스팬을 생성할 수 있으며, 각 스팬에는 고유한 Span ID가 있습니다. 이는 상위 요청 내 도구 호출이나 메모리 이벤트 같은 세부 단계를 나타냅니다.

**MCP 핸드셰이크와 도구 목록 요청을 보여 주는 순서:**

3개의 Trace ID와 각각에 해당하는 Request ID 및 Span ID가 있습니다.

<font color="blue">Trace ID: 1 -> MCP 핸드셰이크 <br/></font>
<font color="purple">Trace ID: 2 -> 핸드셰이크 이후 준비 상태 - notifications/initialized <br/></font>
<font color="green">Trace ID: 3 -> Tools/list 요청/응답<br/></font>

![전체 순서](images/20-traceids-sequence.png)


**Trace ID: 68eb191773c47c5f6babe5cb4f81bd69**

* <font color="blue">Client → Gateway: "Hi, I'm client MCP v0.1.0, can we talk using protocol version 2025-06-18"</font> <br/><br/>
<img src="images/3-sequence1.png" width="60%">
* <font color="blue">Gateway → Client: "Yes, I can talk to you."</font> <br/><br/>
<img src="images/4-sequence2.png" width="60%">
* <font color="blue"> Gateway → Client: "This is my info" </font> <br/><br/>
<img src="images/5-sequence3.png" width="60%">

**Trace ID: 68eb1917268b63ef1290ed317254df8c**
* <font color="purple">Client → Gateway: "After successful initialization, the client sends a notification to indicate it’s ready:"</font> <br/><br/>
<img src="images/6-sequence4.png" width="60%">
* <font color="purple">Gateway → Client: "Received notification about system being ready"</font> <br/><br/>
<img src="images/7-sequence5.png" width="60%">

**Trace ID: 68eb19177631f5357c06be8c182a99df**
* <font color="green">Client → Gateway: "Calling list/tools"</font> <br/><br/>
<img src="images/8-sequence6.png" width="60%">
* <font color="green">Gateway → Client: "Received tools list request"</font> <br/><br/>
<img src="images/9-sequence7.png" width="60%">
* <font color="green">Gateway → Client: "Here are your tools: get_order_tool and update_order_tool"</font> (전체 응답)<br/><br/>
<img src="images/10-sequence8.png" width="60%">

### Console을 사용하여 CloudWatch로 추적 전송 구성

이 섹션에서는 애플리케이션을 통과하는 상호 작용의 흐름을 추적하도록 CloudWatch 추적 전송을 활성화하는 방법을 설명합니다. 이를 통해 요청을 시각화하고, 성능 병목 현상을 식별하고, 오류를 해결하고, 성능을 최적화할 수 있습니다.

#### 1단계: 추적 전송 소스 생성

In [ ]:
traces_source_response = logs_client.put_delivery_source(
    name=f"{gateway_name}-traces-source",
    logType="TRACES",
    resourceArn=create_response["gatewayArn"],
)

#### 2단계: 전송 대상 생성 

In [ ]:
# 전송 대상은 CloudWatch Logs의 로그 그룹, Amazon S3 버킷, Firehose 전송 스트림 또는 X-Ray를 나타낼 수 있으며 여기서는 CloudWatch 로그 그룹 사용

traces_destination_response = logs_client.put_delivery_destination(
    name=f"{gateway_name}-traces-destination", deliveryDestinationType="XRAY"
)
print(traces_destination_response["deliveryDestination"]["arn"])

#### 3단계: 추적 전송 생성(소스와 대상 연결)

In [ ]:
# 소스와 대상을 연결하여 로그 전송 생성. 전송은 이미 생성한 논리 전송 소스와 논리 전송 대상 간의 연결
delivery_response = logs_client.create_delivery(
    deliverySourceName=f"{gateway_name}-traces-source",
    deliveryDestinationArn=traces_destination_response["deliveryDestination"]["arn"],
)

In [ ]:
import time

time.sleep(10)

#### 4단계: AWS Console에서 확인

* AWS Console에서 [Amazon Bedrock AgentCore](https://console.aws.amazon.com/bedrock-agentcore/) 서비스로 이동합니다.
* AWS 리전이 올바른지 확인합니다.
* **Gateways**를 선택합니다.
* 생성한 Gateway를 선택합니다.
* **Tracing**이 `Enabled` 상태인지 확인합니다.

![추적 활성화](images/26-enable-tracing.png)

### GenAI Observability Dashboard의 추적 - Gateway 수준

#### AgentCore Gateway에서 tools/list 작업 호출

In [ ]:
from strands.tools.mcp.mcp_client import MCPClient


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

# 특정 도구 파라미터 정의
TOOL_NAME = "LambdaUsingSDK___get_order_tool"
ORDER_ID = "123"

with client:
    try:
        # get_order_tool 호출
        result = client.call_tool_sync(
            tool_use_id="get-order-id-123-call-1",
            name=TOOL_NAME,
            arguments={"orderId": ORDER_ID},
        )

        # 도구 응답 출력
        print(f"\nGet Order Tool Response for Order ID {ORDER_ID}:")
        print(f"Content: {result['content'][0]['text']}")

    except Exception as e:
        print(f"Error occurred while calling {TOOL_NAME}: {str(e)}")

#### Amazon CloudWatch Gateway 추적 및 스팬

<font color="red"> **참고: 아래 GenAI Observability Dashboard에 Gateway와 추적이 표시되기까지 최소 2~5분이 걸립니다.** </font>

In [ ]:
print(cloudwatch_log_group)

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다.
* **Log Groups**로 이동합니다.
* 위 출력에 표시된 로그 그룹을 선택합니다. 예: **/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/gatewayID**
* `Search all log streams`를 선택합니다.
* CloudWatch 로그에서 `LambdaUsingSDK___get_order_tool` 키워드를 검색합니다. <br/>
* 아래 스크린샷에 표시된 작업에 해당하는 Trace ID를 확인하고 기록해 둡니다.

예: **68ed6dc01c0556e2735177ed3794422a**

![GenAI CloudWatch 로그](images/12-cloudwatch-logs-mcptool-2.png)

* **GenAI Observability** -> **Bedrock AgentCore**로 이동합니다.
* **Gateways**를 선택합니다.

![GenAI 관찰성 Gateway](images/11-cloudwatch-gateways.png)

* **Traces**를 선택하고 Trace ID **68ed6dc01c0556e2735177ed3794422a**를 검색합니다.
* **Trace ID**를 클릭하여 스팬과 지연 시간을 확인합니다. 이 예에서는 `InvokeTool` 작업에 580ms가 걸렸고 평균 스팬 지연 시간은 290ms입니다.

![GenAI 관찰성 추적](images/13-cloudwatch-genai-traces-span.png)

**참고: 추적이 보이지 않으면 오른쪽 상단의 시간 범위를 적절히 조정해야 할 수 있습니다.**

![시간 범위](images/25-bedrock-timewindow.png)

* 추적에서 아래로 더 스크롤하여 `span metadata`의 세부 정보를 확인합니다. <br/>

   `kind:SERVER` - 전체 실행 세부 정보, 호출된 도구, Gateway 세부 정보, AWS Request ID, Trace ID 및 Span ID를 추적합니다.<br/>
   `kind:CLIENT` - 호출된 특정 대상과 대상 유형, 대상 실행 시간, 대상 실행 시작 및 종료 시간 등의 세부 정보를 포함합니다.<br/>

  아래 스크린샷에서는 `AgentCore.Gateway.InvokeTool` 아래에서 도구 실행에 378ms(`execute_tool_latency_ms`)가 걸렸고 도구 실행 시간을 제외한 Gateway 처리 시간은 152ms(`overhead_latency`)였음을 보여 줍니다.

![스팬 메타데이터](images/16-span-metadata.png)

### Amazon CloudWatch를 사용하여 문제의 근본 원인 탐지

**시나리오:** Gateway에 유효하지 않은 토큰을 전송하고 로그와 스팬을 확인합니다.

#### 유효하지 않은 토큰 설정

In [ ]:
token1 = "12345"

In [ ]:
print(cloudwatch_log_group)

#### CloudWatch Logs의 Live Tail 시작

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다.
* **Log Groups**로 이동합니다.
* 위 출력에 표시된 로그 그룹을 선택합니다. 예: **/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/gatewayID**
* **Start tailing**을 클릭합니다.

![CloudWatch Live Tail 화면](images/2-cloudwatch-live-tail.png)

#### 유효하지 않은 토큰으로 특정 Gateway 도구 호출

In [ ]:
from strands.tools.mcp.mcp_client import MCPClient


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token1}"})


client = MCPClient(create_streamable_http_transport)

# 특정 도구 파라미터 정의
TOOL_NAME = "LambdaUsingSDK___get_order_tool"
ORDER_ID = "123"

with client:
    try:
        # get_order_tool 호출
        result = client.call_tool_sync(
            tool_use_id="get-order-id-123-call-1",
            name=TOOL_NAME,
            arguments={"orderId": ORDER_ID},
        )

        # 도구 응답 출력
        print(f"\nGet Order Tool Response for Order ID {ORDER_ID}:")
        print(f"Content: {result['content'][0]['text']}")

    except Exception as e:
        print(f"Error occurred while calling {TOOL_NAME}: {str(e)}")

#### Amazon CloudWatch Logs에서 추가 정보 확인

생성된 예외는 **문제의 근본 원인에 대한 유용한 정보를 제공하지 않습니다.**

    raise MCPClientInitializationError("the client initialization failed") from e
    strands.types.exceptions.MCPClientInitializationError: the client initialization failed


예외의 근본 원인을 파악하려면 CloudWatch 로그를 확인합니다.

![근본 원인](images/17-rootcause.png)

#### 추적 및 스팬에서 추가 정보 확인

* 위 로그에서 `trace_id` 값을 기록해 둡니다.
* CloudWatch Console에서 **GenAI Observability** -> **Bedrock AgentCore**로 이동합니다.
* **Gateways**를 선택합니다.
* **Traces**를 선택하고 Trace ID **68ee6edf04de45005c702392328eb065**를 검색합니다.
* 아래 스크린샷과 같이 스팬에서 자세한 정보를 확인할 수 있습니다.

![문제 해결 추적](images/18-troubleshooting-traces1.png)

### AgentCore Gateway CloudWatch 지표

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다.
* **Metrics** -> **All metrics**를 선택합니다.
* **AWS namespaces** -> **Bedrock-AgentCore**를 선택합니다.
* **Method, Name, Operation, Protocol, Resource**를 선택합니다. Operation 수준에서 집계하려면 **Method, Operation, Protocol, Resource**를 선택할 수 있습니다.
* 원하는 지표를 선택하여 대시보드를 생성하거나 값을 확인합니다.

![지표](images/19-metrics.png)

### 에이전트 추적 이해 - AgentCore Runtime의 Strands Agent에서 AgentCore Gateway로 연결

#### 아래 필수 변수의 값을 큰따옴표와 함께 기록

In [ ]:
print('mcp_url: "%s"' % gatewayURL)
print('user_pool_id: "%s"' % user_pool_id)
print('client_id: "%s"' % client_id)
print('client_secret: "%s"' % client_secret)
print('scopeString: "%s"' % scopeString)
token_endpoint = f"https://{user_pool_id.replace('_', '')}.auth.{REGION}.amazoncognito.com/oauth2/token"
print('token_endpoint: "%s"' % token_endpoint)
print('region: "%s"' % REGION)

#### AgentCore Runtime을 통해 배포할 Bedrock Agent 코드

<font color="red"> 코드에서 **mcp_url**, **user_pool_id**, **client_id**, **client_secret**, **scopeString**, **token_endpoint**, **region** 값을 바꾸세요.</font>

아래 Python 코드는 Model Context Protocol (MCP)을 통해 외부 도구에 액세스하고 사용할 수 있도록 AWS Bedrock AgentCore Gateway에 연결하는 Strands Agent를 생성합니다. 에이전트는 AWS Cognito 자격 증명으로 인증하고, Gateway에서 사용 가능한 도구를 가져오며, Claude 언어 모델을 통해 해당 도구를 호출하여 사용자 쿼리를 처리합니다.

In [ ]:
%%writefile strands_agent_gateway.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from datetime import datetime
from strands import Agent, tool
import logging
from strands.models import BedrockModel
from strands.tools import mcp
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
import os
import time
import functools
import asyncio
import json
import argparse

#### 아래 값을 위 출력의 값으로 교체:

# 게이트웨이 URL:
mcp_url = #<mcp_url> # 이 값을 교체하세요. 예: "https://1demogatewayforlambda-tpwablbixre.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp"
# # Cognito 파라미터:
user_pool_id =  #<user_pool_id> # 이 값을 교체하세요. 예: "us-west-2_CpyXraYjW"
client_id = #<client_id> # 이 값을 교체하세요. 예: "5ifm4heh1r3oa19r2mnngsvung"
client_secret =  #<client_secret> # 이 값을 교체하세요. 예: "kchfnio0inegso43jrc2js0ntgqc5ku2uplhd9v7vp0bo8sp1g0"
scopeString =  #<scopeString> # 이 값을 교체하세요. 예: "sample-agentcore-gateway-id/gateway:read sample-agentcore-gateway-id/gateway:write"
token_endpoint =  #<token_endpoint> # 이 값을 교체하세요. 예: "https://us-west-2_CpyXraYjW.auth.us-west-2.amazoncognito.com/oauth2/token"
region =  #<region> # 이 값을 교체하세요. 예: "us-west-2"

###########################################

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
bedrockmodel = BedrockModel(
    inference_profile_id= model_id,
    temperature=0,
    streaming=True,
)

# 클라이언트를 GatewayClient로 정의
client = GatewayClient(region_name=region)
client.logger.setLevel(logging.DEBUG)

# 토큰 가져오기
client_config = {
    "user_pool_id": user_pool_id,
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": scopeString,
    "region": region,
    "token_endpoint": token_endpoint
}

token_response = client.get_access_token_for_cognito(client_config)
access_token = token_response
print(access_token)


# Gateway 도구 가져오기
def create_streamable_http_transport(mcp_url: str, access_token: str):
    return streamablehttp_client(mcp_url, headers={"Authorization": f"Bearer {access_token}"})
  
def get_full_tools_list(client):
    more_tools = True
    tools = []
    pagination_token = None
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)
        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True 
            pagination_token = tmp_tools.pagination_token
    return tools

def run_agent(mcp_url: str, access_token: str, user_message: str):
    try:
        mcp_client = MCPClient(lambda: create_streamable_http_transport(mcp_url, access_token))
        #region="us-east-1"
        
        with mcp_client:
            tools = get_full_tools_list(mcp_client)
            print(f"Found the following tools: {[tool.tool_name for tool in tools]}")
            agent = Agent(model=bedrockmodel,tools=tools, callback_handler=None)
            print("\nThinking...\n")
            print(user_message)
            result = agent(user_message)        
        return result            
    except Exception as e:
        print(f"Error in run_agent: {e}")
        return {"message": f"Error processing request: {str(e)}"}
        
        
# 미리 빌드된 BedrockAgentCoreApp을 참조하여 앱 정의
app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    try:
        """Process user input and return a response"""
        user_message = payload.get("prompt", "Hello")

        result = run_agent(mcp_url, access_token, user_message)
        print(result)
        return {"result": result.message}
    except Exception as e:
        return {"result": f"Error: {str(e)}"}

if __name__ == "__main__":
    app.run()

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_demo_agent"
response = agentcore_runtime.configure(
    entrypoint="strands_agent_gateway.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)
response

In [ ]:
launch_result = agentcore_runtime.launch()

In [ ]:
import time

time.sleep(20)

#### "Memory is still provisioning (current status: CREATING). Short-term memory takes 30-90 seconds to activate." 오류가 표시되면 몇 초 더 기다린 후 다시 시도하세요.

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "list all tools"})
invoke_response

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "Check the order information for order id 123"})
invoke_response

#### Amazon CloudWatch에서 추적 확인 - 에이전트 수준

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다.
* **GenAI Observability** -> **Bedrock AgentCore**로 이동합니다.
* **Agents** -> **Traces**를 선택합니다.
* 프롬프트마다 하나씩 **두 개의 추적**이 있습니다(list/tools용 추적 하나와 특정 도구 호출용 추적 하나).

**참고: 추적이 보이지 않으면 오른쪽 상단의 시간 범위를 적절히 조정해야 할 수 있습니다.**

![시간 범위](images/25-bedrock-timewindow.png)

![추적](images/14-traces-agentlevel-1.png)
![추적](images/27-invoketools-agent-span.png)

#### 추적의 스팬 확인

![추적](images/15-spans-agent-1.png)

## AWS CloudTrail을 사용한 관찰성

#### CloudTrail 로그 저장용 고유 S3 버킷 이름 생성

In [ ]:
import boto3
import uuid


def generate_s3_bucket_name(base_name):
    account_id = boto3.client("sts").get_caller_identity()["Account"]
    unique_id = uuid.uuid4().hex[:8]
    return f"{base_name}-{account_id}-{unique_id}".lower()

#### S3 버킷 생성 함수

In [ ]:
import boto3


def create_bucket(bucket_name, region):
    try:
        s3_client = boto3.client("s3", region_name=region)
        if region == "us-east-1":
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            location = {"LocationConstraint": region}
            s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=location)
        print(f"Bucket '{bucket_name}' created in region '{region}'")
        return True
    except ClientError as e:
        print(f"Error creating bucket: {e}")
        return False

#### CloudTrail 로그 저장을 허용하는 S3 버킷 정책 생성 함수

In [ ]:
import boto3
import json


def put_bucket_policy(bucket_name, account_id, region, trail_name):
    s3_client = boto3.client("s3")
    trail_arn = f"arn:aws:cloudtrail:{region}:{account_id}:trail/{trail_name}"
    bucket_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AWSCloudTrailAclCheck20150319",
                "Effect": "Allow",
                "Principal": {"Service": "cloudtrail.amazonaws.com"},
                "Action": "s3:GetBucketAcl",
                "Resource": f"arn:aws:s3:::{bucket_name}",
                "Condition": {"StringEquals": {"aws:SourceArn": trail_arn}},
            },
            {
                "Sid": "AWSCloudTrailWrite20150319",
                "Effect": "Allow",
                "Principal": {"Service": "cloudtrail.amazonaws.com"},
                "Action": "s3:PutObject",
                "Resource": f"arn:aws:s3:::{bucket_name}/AWSLogs/{account_id}/*",
                "Condition": {
                    "StringEquals": {
                        "s3:x-amz-acl": "bucket-owner-full-control",
                        "aws:SourceArn": trail_arn,
                    }
                },
            },
        ],
    }
    policy_string = json.dumps(bucket_policy)
    s3_client.put_bucket_policy(Bucket=bucket_name, Policy=policy_string)
    print(f"Bucket policy set for bucket '{bucket_name}' with trail '{trail_name}'.")

#### CloudTrail 로그용 CloudWatch Log Group 생성 함수

In [ ]:
def create_cloudwatch_log_group(log_group_name, account_id, region):
    """CloudWatch Logs 그룹과 CloudTrail 로그 스트림을 생성합니다."""
    logs_client = boto3.client("logs")
    log_stream_name = f"{account_id}_CloudTrail_{region}"

    try:
        # 로그 그룹 생성
        logs_client.create_log_group(logGroupName=log_group_name)
        print(f"Created CloudWatch Logs group: {log_group_name}")

        # 로그 스트림 생성
        logs_client.create_log_stream(logGroupName=log_group_name, logStreamName=log_stream_name)
        print(f"Created CloudWatch Logs stream: {log_stream_name}")

    except logs_client.exceptions.ResourceAlreadyExistsException as e:
        if "Log Group" in str(e):
            print(f"CloudWatch Logs group already exists: {log_group_name}")
            # 그룹이 이미 있어도 로그 스트림 생성 시도
            try:
                logs_client.create_log_stream(logGroupName=log_group_name, logStreamName=log_stream_name)
                print(f"Created CloudWatch Logs stream: {log_stream_name}")
            except logs_client.exceptions.ResourceAlreadyExistsException:
                print(f"CloudWatch Logs stream already exists: {log_stream_name}")
        else:
            print(f"CloudWatch Logs stream already exists: {log_stream_name}")


# 메인 코드에서 사용
account_id = boto3.client("sts").get_caller_identity()["Account"]
trail_name = "AgentCoreGatewayMgmtTrail"
log_group_name = f"/aws/cloudtrail/{trail_name}"

#### CloudTrail용 IAM 역할 생성 함수

In [ ]:
def create_cloudwatch_role(role_name, account_id, region, log_group_name):
    """CloudTrail에서 CloudWatch Logs로 전송할 IAM 역할을 생성합니다."""
    iam = boto3.client("iam")

    # CloudTrail 신뢰 정책
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "cloudtrail.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }

    # 특정 로그 그룹 이름을 사용하는 CloudWatch Logs 정책
    role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AWSCloudTrailCreateLogStream2014110",
                "Effect": "Allow",
                "Action": ["logs:CreateLogStream"],
                "Resource": [
                    f"arn:aws:logs:{region}:{account_id}:log-group:{log_group_name}:log-stream:{account_id}_CloudTrail_{region}*"
                ],
            },
            {
                "Sid": "AWSCloudTrailPutLogEvents20141101",
                "Effect": "Allow",
                "Action": ["logs:PutLogEvents"],
                "Resource": [
                    f"arn:aws:logs:{region}:{account_id}:log-group:{log_group_name}:log-stream:{account_id}_CloudTrail_{region}*"
                ],
            },
        ],
    }

    try:
        # 역할 생성
        role = iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust_policy))

        # 정책 연결
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName=f"{role_name}-policy",
            PolicyDocument=json.dumps(role_policy),
        )

        # 역할을 사용할 수 있을 때까지 잠시 대기
        import time

        time.sleep(5)

        return role["Role"]["Arn"]
    except iam.exceptions.EntityAlreadyExistsException:
        return iam.get_role(RoleName=role_name)["Role"]["Arn"]

In [ ]:
account_id = boto3.client("sts").get_caller_identity()["Account"]
trail_name = "AgentCoreGatewayMgmtTrail"
role_name = f"CloudTrail-{trail_name}-{REGION}"
print(role_name)
log_group_name = f"/aws/cloudtrail/{trail_name}"
print(log_group_name)

# CloudWatchRole 생성
cloudwatch_role_arn = create_cloudwatch_role(role_name, account_id, REGION, log_group_name)
print(cloudwatch_role_arn)

#### IAM 역할이 전파될 때까지 대기

In [ ]:
import time

time.sleep(20)

#### S3 버킷 및 CloudWatch Log Group 생성

In [ ]:
import boto3

# CloudTrail 로그용 S3 버킷 생성
s3_bucket_name = generate_s3_bucket_name("agentcore-gateway")
print(s3_bucket_name)
create_bucket(s3_bucket_name, REGION)
put_bucket_policy(s3_bucket_name, account_id, REGION, trail_name)

# CloudWatch Logs 그룹 생성
create_cloudwatch_log_group(log_group_name, account_id, REGION)

In [ ]:
import time

time.sleep(20)

#### AgentCore Gateway 관리 이벤트 기록용 CloudTrail 생성

In [ ]:
# CloudTrail 생성
cloudtrail_client = boto3.client("cloudtrail", region_name=REGION)

# 추적 생성
response = cloudtrail_client.create_trail(
    Name=trail_name,
    S3BucketName=s3_bucket_name,
    CloudWatchLogsLogGroupArn=f"arn:aws:logs:{REGION}:{account_id}:log-group:{log_group_name}:*",
    CloudWatchLogsRoleArn=cloudwatch_role_arn,
)

# AgentCore Gateway 관리 이벤트만 포함하도록 고급 이벤트 선택기 정의
advanced_event_selectors = [
    {
        "Name": "AgentCoreGatewayManagementEvents",
        "FieldSelectors": [{"Field": "eventCategory", "Equals": ["Management"]}],
    }
]

# 고급 이벤트 선택기를 사용하도록 추적 업데이트
cloudtrail_client.put_event_selectors(TrailName=trail_name, AdvancedEventSelectors=advanced_event_selectors)

# 이벤트 로깅 시작
cloudtrail_client.start_logging(Name=trail_name)

print(f"CloudTrail trail '{trail_name}' created and logging management events for AgentCore Gateway.")

#### AWS Console에서 확인

* AWS Console에서 [AWS CloudTrail]((https://console.aws.amazon.com/cloudtrailv2/)) 서비스로 이동합니다.
* AWS 리전이 올바른지 확인합니다.
* Trails를 클릭하고 `AgentCoreGatewayMgmtTrail`이 정상적으로 생성되었는지 확인합니다.

![관리 추적](images/28-cloudtrail.png)


In [ ]:
import time

time.sleep(20)

#### tools/list 호출 및 CloudTrail 이벤트에서 API 추적 확인

In [ ]:
def list_gateways():
    """모든 Bedrock AgentCore gateway를 나열합니다."""
    try:
        # Bedrock AgentCore 클라이언트 초기화
        gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

        print(REGION)

        # Gateway 목록 조회
        response = gateway_client.list_gateways()
        print(response)

        # Gateway 세부 정보 출력
        if "items" in response and response["items"]:
            print("\nGateways found:")
            for gateway in response["items"]:
                print(f"\nName: {gateway.get('name')}")
                print(f"Gateway ID: {gateway.get('gatewayId')}")
                print(f"Gateway URL: {gateway.get('gatewayUrl')}")
                print(f"Status: {gateway.get('status')}")
                print(f"Protocol Type: {gateway.get('protocolType')}")
                print(f"Authorizer Type: {gateway.get('authorizerType')}")
        else:
            print("No gateways found")

        return response.get("gateways", [])

    except Exception as e:
        print(f"Error listing gateways: {str(e)}")
        return []


# 함수 호출
gateways = list_gateways()

In [ ]:
import time

time.sleep(20)

<font color="red"> CloudTrail 로그에 로그 항목이 표시되기까지 몇 초 정도 걸릴 수 있습니다.</font>

#### CloudTrail 이벤트 확인

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다. <br/>
* CloudTrail Log Groups -> `/aws/cloudtrail/AgentCoreGatewayMgmtTrail` -> `Search all log streams`를 선택합니다. <br/>
* 검색 상자에서 `ListGateways`를 검색합니다.

**`IAMUser` 유형의 ListGateways API 호출** <br/><br/>
![ListGateways API 호출](images/22-list-gateways-1.png)

CreateGateway 작업과 기타 [Gateway 이벤트](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-event-types.html)도 마찬가지로 관리 이벤트 아래에 기록됩니다. <br/> <br/>
![관리 이벤트](images/21-cloudtrail-mgmt.png)

#### CloudTrail에 데이터 이벤트 기록

**<font color="red"> 데이터 이벤트는 발생량이 많아 기본적으로 활성화되지 않으며, 활성화하면 추가 비용이 발생합니다.</font>**

In [ ]:
account_id = boto3.client("sts").get_caller_identity()["Account"]
trail_name = "AgentCoreGatewayDataTrail"
role_name = f"CloudTrail-{trail_name}-{REGION}"
print(role_name)
log_group_name = f"/aws/cloudtrail/{trail_name}"
print(log_group_name)

# CloudWatchRole 생성
cloudwatch_role_arn = create_cloudwatch_role(role_name, account_id, REGION, log_group_name)
print(cloudwatch_role_arn)

In [ ]:
import time

time.sleep(20)

In [ ]:
import boto3

# CloudTrail 로그용 S3 버킷 생성
s3_bucket_name = generate_s3_bucket_name("agentcore-gateway-data")
print(s3_bucket_name)
create_bucket(s3_bucket_name, REGION)
put_bucket_policy(s3_bucket_name, account_id, REGION, trail_name)

# CloudWatch Logs 그룹 생성
create_cloudwatch_log_group(log_group_name, account_id, REGION)

In [ ]:
import time

time.sleep(20)

In [ ]:
# CloudTrail 생성
cloudtrail_client = boto3.client("cloudtrail", region_name=REGION)

# 추적 생성
response = cloudtrail_client.create_trail(
    Name=trail_name,
    S3BucketName=s3_bucket_name,
    CloudWatchLogsLogGroupArn=f"arn:aws:logs:{REGION}:{account_id}:log-group:{log_group_name}:*",
    CloudWatchLogsRoleArn=cloudwatch_role_arn,
)

# AgentCore Gateway 관리 이벤트만 포함하도록 고급 이벤트 선택기 정의
advanced_event_selectors = [
    {
        "Name": "AgentCoreGatewayDataEvents",
        "FieldSelectors": [
            {"Field": "eventCategory", "Equals": ["Data"]},
            {
                "Field": "resources.type",
                "Equals": ["AWS::BedrockAgentCore::Gateway"],  # 수정된 서비스 이름
            },
        ],
    }
]

# 고급 이벤트 선택기를 사용하도록 추적 업데이트
cloudtrail_client.put_event_selectors(TrailName=trail_name, AdvancedEventSelectors=advanced_event_selectors)

# 이벤트 로깅 시작
cloudtrail_client.start_logging(Name=trail_name)

print(f"CloudTrail trail '{trail_name}' created and logging data events for AgentCore Gateway.")

#### AWS Console에서 확인

* AWS Console에서 [AWS CloudTrail]((https://console.aws.amazon.com/cloudtrailv2/)) 서비스로 이동합니다.
* AWS 리전이 올바른지 확인합니다.
* Trails를 클릭하고 `AgentCoreGatewayDataTrail`이 정상적으로 생성되었는지 확인합니다.


![관리 추적](images/29-data-cloudtrail1.png)

In [ ]:
import time

time.sleep(20)

#### AgentCore Gateway용 토큰 가져오기

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes"
)
token_response = utils.get_token(user_pool_id, client_id, client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

In [ ]:
from strands.tools.mcp.mcp_client import MCPClient

print(gatewayURL)


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

# 특정 도구 파라미터 정의
TOOL_NAME = "LambdaUsingSDK___get_order_tool"
ORDER_ID = "123"

with client:
    try:
        # get_order_tool 호출
        result = client.call_tool_sync(
            tool_use_id="get-order-id-123-call-1",
            name=TOOL_NAME,
            arguments={"orderId": ORDER_ID},
        )

        # 도구 응답 출력
        print(f"\nGet Order Tool Response for Order ID {ORDER_ID}:")
        print(f"Content: {result['content'][0]['text']}")

    except Exception as e:
        print(f"Error occurred while calling {TOOL_NAME}: {str(e)}")

#### CloudTrail 이벤트 확인

* **[Amazon CloudWatch Console](https://console.aws.amazon.com/cloudwatch/home)**로 이동합니다. <br/>
* CloudTrail Log Groups -> `/aws/cloudtrail/AgentCoreGatewayDataTrail` -> `Search all log streams`를 선택합니다. <br/>

<font color="red"> CloudTrail 로그에 로그 항목이 표시되기까지 몇 초 정도 걸릴 수 있습니다.</font>

![CloudTrail 데이터](images/23-cloudtrail-data.png)

# 정리

이 스크립트는 생성된 리소스를 거의 모두 삭제합니다. 일부 리소스가 삭제되지 않거나 삭제에 실패하면 수동으로 삭제해야 할 수 있습니다. 

In [ ]:
import boto3
import os
import time

# AWS 클라이언트 초기화
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
logs_client = boto3.client("logs", region_name=REGION)
cloudtrail_client = boto3.client("cloudtrail", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)
iam_client = boto3.client("iam", region_name=REGION)
cognito_client = boto3.client("cognito-idp", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
ecr_client = boto3.client("ecr", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)

account_id = sts_client.get_caller_identity()["Account"]


def get_agentcore_runtime_id_by_name(agent_name, region):
    """
    에이전트 이름으로 AgentCore Runtime ID를 조회합니다.

    :param agent_name: 찾을 agent runtime 이름
    :param region: runtime이 있는 AWS 리전
    :return: 찾은 runtime ID, 없으면 None
    """
    agent_runtime_id = None

    try:
        client = boto3.client("bedrock-agentcore-control", region_name=region)
        response = client.list_agent_runtimes()
        print(f"Searching for agent runtime matching: {agent_name}")

        for runtime in response.get("agentRuntimes", []):
            if runtime.get("agentRuntimeId", "").find(agent_name) >= 0:
                agent_runtime_id = runtime.get("agentRuntimeId")
                print(f"Found runtime ID: {agent_runtime_id}")
                print(f"Found runtime: {agent_name}")
                break

        return agent_runtime_id

    except ClientError as e:
        print(f"Error listing runtimes: {e}")
        raise


def delete_agentcore_runtime(region, agent_runtime_id):
    """올바른 runtime ID를 사용해 AgentCore runtime을 삭제합니다."""
    try:
        client = boto3.client("bedrock-agentcore-control", region_name=region)
        response = client.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
        print(f"Successfully deleted AgentCore Runtime: {agent_runtime_id}")
        return response
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException":
            print(f"AgentCore runtime {agent_runtime_id} not found - may already be deleted")
        elif error_code == "ConflictException":
            print(f"Cannot delete AgentCore runtime {agent_runtime_id} - resource is in use")
        elif error_code == "AccessDeniedException":
            print("Access denied. Ensure caller has bedrock-agentcore:DeleteAgentRuntime permission")
        else:
            print(f"Error deleting AgentCore runtime: {e}")
        raise


def delete_ecr_repository(ecr_repository_name):
    """이름이 제공되면 ECR repository를 삭제합니다."""
    try:
        print(f"Deleting ECR repository: {ecr_repository_name}")
        ecr_response = ecr_client.delete_repository(
            repositoryName=ecr_repository_name,
            force=True,  # 이미지가 있어도 강제 삭제
        )
        print(f"ECR repository {ecr_repository_name} deleted successfully")
        return ecr_response
    except ClientError as e:
        print(f"Error deleting ECR repository {ecr_repository_name}: {e}")
        raise


def delete_gateway_and_targets(gateway_id):
    """gateway target과 gateway 자체를 삭제합니다."""
    try:
        # 모든 Gateway 대상 조회 및 삭제
        targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id)
        for target in targets_response.get("items", []):
            target_id = target["targetId"]
            print(f"Deleting gateway target: {target_id}")
            gateway_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetIdentifier=target_id)
            time.sleep(2)

        # Gateway 삭제
        print(f"Deleting gateway: {gateway_id}")
        gateway_client.delete_gateway(gatewayIdentifier=gateway_id)
        print(f"Gateway {gateway_id} deleted successfully")

    except ClientError as e:
        print(f"Error deleting gateway: {e}")


def delete_cloudwatch_log_deliveries(gateway_name):
    """CloudWatch 로그 및 trace delivery를 삭제합니다."""
    try:
        logs_source_name = f"{gateway_name}-logs-source"
        traces_source_name = f"{gateway_name}-traces-source"

        # 전송 목록 조회 및 삭제
        deliveries = logs_client.describe_deliveries()
        for delivery in deliveries.get("deliveries", []):
            delivery_id = delivery["id"]
            print(f"Deleting delivery: {delivery_id}")
            logs_client.delete_delivery(id=delivery_id)
            time.sleep(1)

        # 전송 소스 삭제
        for source_name in [logs_source_name, traces_source_name]:
            try:
                print(f"Deleting delivery source: {source_name}")
                logs_client.delete_delivery_source(name=source_name)
            except ClientError as e:
                print(f"Error deleting source {source_name}: {e}")

        # 전송 대상 삭제
        destinations = logs_client.describe_delivery_destinations()
        for dest in destinations.get("deliveryDestinations", []):
            dest_name = dest["name"]
            if "bedrock-agentcore-gw" in dest_name or gateway_name in dest_name:
                print(f"Deleting delivery destination: {dest_name}")
                logs_client.delete_delivery_destination(name=dest_name)
                time.sleep(1)

    except ClientError as e:
        print(f"Error deleting log deliveries: {e}")


def delete_cloudwatch_log_groups(gateway_id):
    """CloudWatch 로그 그룹을 삭제합니다."""
    log_groups = [
        f"/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{gateway_id}",
        "/aws/cloudtrail/AgentCoreGatewayMgmtTrail",
        "/aws/cloudtrail/AgentCoreGatewayDataTrail",
    ]

    for log_group in log_groups:
        try:
            print(f"Deleting log group: {log_group}")
            logs_client.delete_log_group(logGroupName=log_group)
            print(f"Log group {log_group} deleted")
        except ClientError as e:
            print(f"Error deleting log group {log_group}: {e}")


def delete_cloudtrail_trails():
    """CloudTrail trail을 삭제합니다."""
    trails = ["AgentCoreGatewayMgmtTrail", "AgentCoreGatewayDataTrail"]

    for trail_name in trails:
        try:
            # 로깅 중지
            print(f"Stopping logging for trail: {trail_name}")
            cloudtrail_client.stop_logging(Name=trail_name)
            time.sleep(2)

            # 추적 삭제
            print(f"Deleting trail: {trail_name}")
            cloudtrail_client.delete_trail(Name=trail_name)
            print(f"Trail {trail_name} deleted")

        except ClientError as e:
            print(f"Error deleting trail {trail_name}: {e}")


def empty_and_delete_s3_bucket(bucket_name):
    """S3 버킷을 비우고 삭제합니다."""
    try:
        # 모든 객체 조회 및 삭제
        print(f"Emptying S3 bucket: {bucket_name}")
        paginator = s3_client.get_paginator("list_objects_v2")

        for page in paginator.paginate(Bucket=bucket_name):
            if "Contents" in page:
                objects = [{"Key": obj["Key"]} for obj in page["Contents"]]
                s3_client.delete_objects(Bucket=bucket_name, Delete={"Objects": objects})

        # 버킷 삭제
        print(f"Deleting S3 bucket: {bucket_name}")
        s3_client.delete_bucket(Bucket=bucket_name)
        print(f"S3 bucket {bucket_name} deleted")

    except ClientError as e:
        print(f"Error deleting S3 bucket {bucket_name}: {e}")


def delete_iam_roles():
    """CloudTrail 및 Gateway용으로 생성한 IAM 역할을 삭제합니다."""
    roles = [
        f"CloudTrail-AgentCoreGatewayMgmtTrail-{REGION}",
        f"CloudTrail-AgentCoreGatewayDataTrail-{REGION}",
        "sample-lambdagateway-role",
    ]

    for role_name in roles:
        try:
            # 인라인 정책 삭제
            policies = iam_client.list_role_policies(RoleName=role_name)
            for policy_name in policies.get("PolicyNames", []):
                print(f"Deleting inline policy {policy_name} from role {role_name}")
                iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)

            # 관리형 정책 분리
            attached_policies = iam_client.list_attached_role_policies(RoleName=role_name)
            for policy in attached_policies.get("AttachedPolicies", []):
                print(f"Detaching policy {policy['PolicyArn']} from role {role_name}")
                iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy["PolicyArn"])

            # 역할 삭제
            print(f"Deleting IAM role: {role_name}")
            iam_client.delete_role(RoleName=role_name)
            print(f"IAM role {role_name} deleted")

        except ClientError as e:
            print(f"Error deleting IAM role {role_name}: {e}")


def delete_cognito_resources(user_pool_id, client_id):
    """Cognito user pool과 client를 삭제합니다."""
    try:
        # 앱 클라이언트 삭제
        print(f"Deleting Cognito app client: {client_id}")
        cognito_client.delete_user_pool_client(UserPoolId=user_pool_id, ClientId=client_id)

        # 리소스 서버 삭제
        resource_server_id = "sample-agentcore-gateway-id"
        print(f"Deleting Cognito resource server: {resource_server_id}")
        cognito_client.delete_resource_server(UserPoolId=user_pool_id, Identifier=resource_server_id)

        # User Pool 삭제
        print(f"Deleting Cognito user pool: {user_pool_id}")
        cognito_client.delete_user_pool(UserPoolId=user_pool_id)
        print(f"Cognito user pool {user_pool_id} deleted")

    except ClientError as e:
        print(f"Error deleting Cognito resources: {e}")


def delete_lambda_function(function_name="gateway_lambda"):
    """Lambda 함수를 삭제합니다."""
    try:
        print(f"Deleting Lambda function: {function_name}")
        lambda_client.delete_function(FunctionName=function_name)
        print(f"Lambda function {function_name} deleted")
    except ClientError as e:
        print(f"Error deleting Lambda function: {e}")


# 기본 정리 작업 실행
if __name__ == "__main__":
    print("=" * 80)
    print("AMAZON BEDROCK AGENTCORE GATEWAY OBSERVABILITY CLEANUP")
    print("=" * 80)

    # 중요: 실제 리소스 식별자로 바꾸기
    gateway_id = gatewayID  # 실제 Gateway ID로 바꾸기
    gateway_name = gatewayID  # 실제 Gateway 이름으로 바꾸기
    user_pool_id = user_pool_id  # 실제 User Pool ID로 바꾸기
    client_id = client_id  # 실제 클라이언트 ID로 바꾸기
    agent_name = "strands_demo_agent"  # 검색할 에이전트 이름
    ecr_repository_name = "bedrock-agentcore-" + agent_name  # 실제 ECR 리포지토리 이름으로 바꾸기(선택 사항)

    # 1단계: AgentCore Runtime Agent 및 ECR 리포지토리 조회 및 삭제
    print("\n[1/9] Looking up and Deleting AgentCore Runtime Agent and ECR Repository...")
    try:
        # 함수를 사용하여 이름으로 Runtime ID 조회
        agent_runtime_id = get_agentcore_runtime_id_by_name(agent_name, REGION)

        if agent_runtime_id:
            print(f"Region: {REGION}")
            # 삭제 함수를 사용하여 Runtime 삭제
            delete_agentcore_runtime(REGION, agent_runtime_id)
            time.sleep(5)

            # ECR 리포지토리 이름이 제공된 경우 삭제
            if ecr_repository_name:
                delete_ecr_repository(ecr_repository_name)
            else:
                print("No ECR repository name provided - skipping ECR cleanup")
        else:
            print(f"No runtime found matching: {agent_name}")
            print("Skipping AgentCore runtime deletion...")
    except Exception as e:
        print(f"Failed to delete AgentCore runtime: {e}")
        print("Continuing with remaining cleanup tasks...")

    # 2단계: Gateway 및 대상 삭제
    print("\n[2/9] Deleting Gateway and Targets...")
    try:
        delete_gateway_and_targets(gateway_id)
        time.sleep(5)
    except Exception as e:
        print(f"Error in gateway deletion: {e}")

    # 3단계: CloudWatch 로그 전송 삭제
    print("\n[3/9] Deleting CloudWatch Log Deliveries...")
    try:
        delete_cloudwatch_log_deliveries(gateway_name)
        time.sleep(5)
    except Exception as e:
        print(f"Error deleting log deliveries: {e}")

    # 4단계: CloudWatch Log Group 삭제
    print("\n[4/9] Deleting CloudWatch Log Groups...")
    try:
        delete_cloudwatch_log_groups(gateway_id)
    except Exception as e:
        print(f"Error deleting log groups: {e}")

    # 5단계: CloudTrail 추적 삭제
    print("\n[5/9] Deleting CloudTrail Trails...")
    try:
        delete_cloudtrail_trails()
        time.sleep(5)
    except Exception as e:
        print(f"Error deleting trails: {e}")

    # 6단계: S3 버킷 삭제
    print("\n[6/9] Deleting S3 Buckets...")
    try:
        buckets = s3_client.list_buckets()
        for bucket in buckets["Buckets"]:
            bucket_name = bucket["Name"]
            if "agentcore-gateway" in bucket_name:
                empty_and_delete_s3_bucket(bucket_name)
    except Exception as e:
        print(f"Error deleting S3 buckets: {e}")

    # 7단계: IAM 역할 삭제
    print("\n[7/9] Deleting IAM Roles...")
    try:
        delete_iam_roles()
    except Exception as e:
        print(f"Error deleting IAM roles: {e}")

    # 8단계: Cognito 리소스 삭제
    print("\n[8/9] Deleting Cognito Resources...")
    try:
        delete_cognito_resources(user_pool_id, client_id)
    except Exception as e:
        print(f"Error deleting Cognito resources: {e}")

    # 9단계: Lambda 함수 삭제
    print("\n[9/9] Deleting Lambda Function...")
    try:
        delete_lambda_function()
    except Exception as e:
        print(f"Error deleting Lambda function: {e}")

    print("\n" + "=" * 80)
    print("CLEANUP PROCESS COMPLETED")
    print("=" * 80)
    print("\nPlease verify in the AWS Console that all resources have been deleted.")
    print("Some resources may take a few minutes to fully delete.")
    print("\nVerification checklist:")
    print("  ✓ Amazon Bedrock AgentCore - Gateways")
    print("  ✓ Amazon Bedrock AgentCore - Runtime")
    print("  ✓ Amazon CloudWatch - Log Groups")
    print("  ✓ AWS CloudTrail - Trails")
    print("  ✓ Amazon S3 - Buckets")
    print("  ✓ AWS IAM - Roles")
    print("  ✓ Amazon Cognito - User Pools")
    print("  ✓ AWS Lambda - Functions")
    print("  ✓ Amazon ECR - Repositories")